# 03 — Modelling & Evaluation
## AI-Driven Predictive Monitoring System for Supply Chain Disruptions

This notebook trains and evaluates all predictive models:

| Model | Task | Algorithm |
|---|---|---|
| **Delay Classifier** | Predict `delayed` (0/1) | XGBoost + scale_pos_weight |
| **Delay Regressor** | Estimate `delay_hours` | XGBoost (log-transformed target) |
| **Congestion Forecaster** | 7-day ahead forecast | LSTM (PyTorch) |
| **Anomaly Detector** | Flag unusual shipments | Isolation Forest + LOF ensemble |

Evaluation protocol:
- Time-based split (train 70% / val 15% / test 15%)
- Classification: AUC-ROC, Average Precision, F1-Macro
- Regression: RMSE, MAE, R²
- SHAP values for model explainability


In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score,
    roc_curve, precision_recall_curve,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)

ROOT     = Path("..").resolve()
DATA_DIR = ROOT / "data"
PROC_DIR = DATA_DIR / "processed"
MDL_DIR  = ROOT / "models"
MDL_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(ROOT))
from src.data.loader import SupplyChainLoader
from src.features.build_features import build_feature_matrix, FEATURE_COLS, TARGET_CLF, TARGET_REG

loader     = SupplyChainLoader(DATA_DIR)
shipments  = loader.shipments()
congestion = loader.port_congestion()

df         = build_feature_matrix(shipments, congestion)
feat_cols  = [c for c in FEATURE_COLS if c in df.columns]
df.fillna(df.median(numeric_only=True), inplace=True)

n      = len(df)
val_c  = int(n * 0.70)
test_c = int(n * 0.85)
train  = df.iloc[:val_c].reset_index(drop=True)
val    = df.iloc[val_c:test_c].reset_index(drop=True)
test   = df.iloc[test_c:].reset_index(drop=True)

print(f"train:{len(train)}  val:{len(val)}  test:{len(test)}")
print(f"Feature cols ({len(feat_cols)}): {feat_cols[:8]} ...")
print(f"Delay rate — train: {train[TARGET_CLF].mean():.3f}  val: {val[TARGET_CLF].mean():.3f}  test: {test[TARGET_CLF].mean():.3f}")


In [ ]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    classification_report, mean_absolute_error,
    mean_squared_error, r2_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data.loader import SupplyChainLoader
from src.features.build_features import build_feature_matrix, FEATURE_COLS
from src.models.train_xgboost import train_classifier, train_regressor, _load_config
from src.anomaly.anomaly_detection import AnomalyDetector

cfg = _load_config()
